In [1]:
import sys
print(sys.executable)

/Users/rahul/Documents/Manisha/My_Projects/enterprise-rag-system/.venv/bin/python


# Embeddings & Semantic Similarity

##### How text is coverted into embeddings and how the semantic Similarity between the Embeddings is measured

In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")
# The all-MiniLM-L6-v2 model is a pre-trained neural network trained to produce meaningful sentence embeddings
#model produces a 384-dimensional embedding, so every sentence is encoded with becomes a vector of exactly 384 numbers.
# all vectors stored in the same vector index need to have the same dimensionality, because the database needs to compare them in the same vector space.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'all-MiniLM-l6-v2' is useful, because it is:
- relatively small
- fast on CPU
- good enough for experimentation
- commonly used for sentence embeddings
- easy to run locally without a GPU

In [4]:
sentences = [
    "Students need to submit their thesis before the deadline",
    "Students can apply for a working student position during their studies",
    "The Submissions are to be done online through the university portal",
    "Working students can work up to 20 hours per week during the semester"
]

In [5]:
embeddings = model.encode(sentences)

In [6]:
print(type(embeddings))

<class 'numpy.ndarray'>


In [7]:
print(embeddings.shape)

(4, 384)


Where,  
4 = number of sentences
384 = embedding dimensions for each sentence.

In [8]:
# One Embedding
print(embeddings[0])

[-2.21754815e-02  2.24780738e-02  1.88459046e-02 -2.53352597e-02
 -1.89582780e-02  3.73594984e-02 -1.34182572e-01 -1.61554816e-03
 -7.90390279e-03  8.58193785e-02  9.53538716e-02  4.56336774e-02
 -5.65632172e-02 -3.91030125e-03 -3.29908803e-02  2.46825945e-02
  1.69996805e-02 -3.88671756e-02  4.32109013e-02  3.53365950e-02
  5.37744239e-02 -2.83664884e-03  1.51708983e-02 -1.52831236e-02
 -6.01277128e-03  3.63258645e-02 -1.28296409e-02 -1.05409727e-01
  1.59105863e-02  2.13803872e-02  1.17253261e-02  6.39921636e-04
  4.54749987e-02  1.74225438e-02  8.13098103e-02  1.46168051e-02
  1.01209264e-02  3.16475593e-02 -6.14539860e-03 -6.02596998e-03
 -8.11066329e-02 -5.56119308e-02 -2.64768861e-02  4.79421429e-02
 -2.13631969e-02 -1.04132546e-02 -3.37646529e-02 -4.75147814e-02
 -1.03577999e-02 -1.48294335e-02 -2.79332437e-02 -5.34145087e-02
 -6.17201924e-02 -7.82495663e-02 -9.53678638e-02 -5.99806532e-02
  5.11051901e-02 -5.20951040e-02  1.68306120e-02 -2.67778225e-02
 -6.46229088e-02  2.17803

In [9]:
print(embeddings)

[[-0.02217548  0.02247807  0.0188459  ...  0.0879171   0.0134713
  -0.01706664]
 [-0.02025273  0.05329028  0.00882855 ... -0.00267111  0.01984229
   0.05962213]
 [-0.01345659 -0.02212814 -0.0038855  ...  0.06466551  0.00807562
   0.01575514]
 [-0.01805258  0.02728637  0.0136765  ... -0.01238019 -0.04973829
  -0.01092467]]


#### Observation:

4 sentences and each sentence is represented by a 384-dimensional vector. 

#### To know:

The important property is the position of the complete vector in the embedding space, which allows semantically similar sentences to be compared based on their vector similarity.

We need a way to measure how similar two embedding vectors are. cosine_similarity measures the angle/direction between two vectors.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

In [11]:
print(similarity_matrix.shape)

print(similarity_matrix)
# similarity scores; A value closer to 1 indicates greater similarity, while lower values indicate less similarity.
# Each sentence is compared with itself. A vector has maximum similarity with itself, so the score is 1.

(4, 4)
[[1.0000001  0.34767598 0.4515454  0.2855668 ]
 [0.34767598 0.9999999  0.34781277 0.51683426]
 [0.4515454  0.34781277 0.9999999  0.17189313]
 [0.2855668  0.51683426 0.17189313 1.        ]]


#### observation- 

The diagonal contains 1 or near to 1 because each sentence is compared with itself.

The matrix currently contains only numbers. Adding the actual sentences as row and column labels lets us understand which two sentences each score belongs to.

In [12]:
import pandas as pd

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=sentences,
    columns=sentences
)

similarity_df

,Students need to submit their thesis before the deadline,Students can apply for a working student position during their studies,The Submissions are to be done online through the university portal,Working students can work up to 20 hours per week during the semester
Students need to submit their thesis before the deadline,1.000000,0.347676,0.451545,0.285567
Students can apply for a working student position during their studies,0.347676,1.000000,0.347813,0.516834
The Submissions are to be done online through the university portal,0.451545,0.347813,1.000000,0.171893
Working students can work up to 20 hours per week during the semester,0.285567,0.516834,0.171893,1.000000


Replace the diagonal with -1 and find the pair of different sentences with the highest similarity.

In [16]:
#Find highest similarity pair

import numpy as np
# replacing the diagonal with -1 temporarily; no self-similarity
np.fill_diagonal(similarity_matrix, -1)  

max_index = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
min_index = np.unravel_index(np.argmin(similarity_matrix), similarity_matrix.shape)

print("Sentence 1(max): ", sentences[max_index[0]])
print("Sentence 2(max): ", sentences[max_index[1]])
print("")
print("Sentence 1(min): ", sentences[min_index[0]])
print("Sentence 2(min): ", sentences[min_index[1]])
print("")
print("Similarity (Higher similarity pair): ", similarity_matrix[max_index])
print("Similarity (Lower similarity pair): ", similarity_matrix[min_index])

Sentence 1(max):  Students can apply for a working student position during their studies
Sentence 2(max):  Working students can work up to 20 hours per week during the semester

Sentence 1(min):  Students need to submit their thesis before the deadline
Sentence 2(min):  Students need to submit their thesis before the deadline

Similarity (Higher similarity pair):  0.51683426
Similarity (Lower similarity pair):  -1.0


#### Observation:

The similarity scores shows that the sentences are compared based on their semantic representations rather than only matching exact words. 
The highest-scoring pair represents the sentences that are most semantically related within this small dataset.

#### Retrieval part:

Asking a user question to the system

In real RAG system, the question is not known beforehand, A user provides a query, and the system needs to find relevant information.

In [18]:
query = "Can students work 20 hours per week?"

#To compare the above query with the existing sentence embeddings, covert the query into an embedding
# then rank and return the most relevant one

In [24]:
#coverting query into embedding
query_embedding = model.encode(query)

print(type(query_embedding))
print(query_embedding.shape)
print(query_embedding[0:10])

<class 'numpy.ndarray'>
(384,)
[-0.01734187  0.06885585  0.03051276  0.04359627 -0.01426481 -0.01954645
 -0.0463654  -0.09656551 -0.10468689  0.00041971]


In [26]:
#Compare the query with all the sentences - information retrieval
# this shows how semantically similar the user question is to each available sentence.

query_similarities = cosine_similarity(query_embedding.reshape(1, -1), embeddings)[0]
print(query_similarities)


[0.25033003 0.4485857  0.12178122 0.8707342 ]


In [ ]:
# rank the results - sort
ranked_indices = np.argsort()